In [2]:
%%file producer.py
from kafka import KafkaProducer
import json, random, time
from datetime import datetime

producer = KafkaProducer(
    bootstrap_servers='broker:9092',
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)

sklepy = ['Warszawa', 'Kraków', 'Gdańsk', 'Wrocław']
kategorie = ['elektronika', 'odzież', 'żywność', 'książki']

def generate_transaction():
    return {
        'tx_id': f'TX{random.randint(1000,9999)}',
        'user_id': f'u{random.randint(1,20):02d}',
        'amount': round(random.uniform(5.0, 5000.0), 2),
        'store': random.choice(sklepy),
        'category': random.choice(kategorie),
        'timestamp': datetime.now().isoformat(),
    }

for i in range(1000):
    tx = generate_transaction()
    producer.send('transactions', value=tx)
    print(f"[{i+1}] {tx['tx_id']} | {tx['amount']:.2f} PLN | {tx['store']}")
    time.sleep(1)

producer.flush()
producer.close()

Overwriting producer.py


In [4]:
%%file consumer_filter.py
from kafka import KafkaConsumer
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

print("Nasłuchuję na duże transakcje (amount > 3000)...")

for message in consumer:
    tx = message.value
    
    if tx['amount'] > 3000:
        print(f"ALERT: {tx['tx_id']} | {tx['amount']:.2f} PLN | {tx['store']} | {tx['category']}")

Overwriting consumer_filter.py


In [5]:
%%file consumer_enrich.py
from kafka import KafkaConsumer
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    group_id='consumer-enrich-group',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

print("Nasłuchuję i wzbogacam transakcje o risk_level...")

for message in consumer:
    tx = message.value
    amount = tx['amount']

    if amount > 3000:
        tx['risk_level'] = 'HIGH'
    elif amount > 1000:
        tx['risk_level'] = 'MEDIUM'
    else:
        tx['risk_level'] = 'LOW'

    print(tx)

Writing consumer_enrich.py


In [6]:
%%file consumer_count.py
from kafka import KafkaConsumer
from collections import Counter
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

store_counts = Counter()
total_amount = {}
msg_count = 0

print("Zliczam transakcje per sklep...")

for message in consumer:
    tx = message.value
    store = tx['store']
    amount = tx['amount']

    store_counts[store] += 1

    if store not in total_amount:
        total_amount[store] = 0
    total_amount[store] += amount

    msg_count += 1

    if msg_count % 10 == 0:
        print("\nSklep | Liczba | Suma | Średnia")
        print("-" * 40)

        for store in store_counts:
            count = store_counts[store]
            total = total_amount[store]
            avg = total / count
            print(f"{store} | {count} | {total:.2f} | {avg:.2f}")

Writing consumer_count.py


In [9]:
%%file consumer_stats.py
from kafka import KafkaConsumer
from collections import defaultdict
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    group_id='consumer-stats-v1',
    auto_offset_reset='earliest',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

stats = defaultdict(lambda: {
    'count': 0,
    'sum': 0.0,
    'min': float('inf'),
    'max': float('-inf')
})

msg_count = 0

print("Zliczam statystyki per kategoria...")

for message in consumer:
    tx = message.value
    category = tx['category']
    amount = tx['amount']

    stats[category]['count'] += 1
    stats[category]['sum'] += amount
    stats[category]['min'] = min(stats[category]['min'], amount)
    stats[category]['max'] = max(stats[category]['max'], amount)

    msg_count += 1
    print(f"Odebrano {msg_count}: {category} | {amount:.2f}")

    if msg_count % 3 == 0:
        print("\nKategoria | Liczba | Suma | Min | Max")
        print("-" * 55)
        for category, s in stats.items():
            print(f"{category} | {s['count']} | {s['sum']:.2f} | {s['min']:.2f} | {s['max']:.2f}")

Overwriting consumer_stats.py


In [10]:
%%file consumer_speed.py
from kafka import KafkaConsumer
from collections import defaultdict, deque
from datetime import datetime
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    group_id='consumer-speed-v1',
    auto_offset_reset='latest',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

user_events = defaultdict(deque)

print("Nasłuchuję anomalii prędkości...")

for message in consumer:
    tx = message.value
    user_id = tx['user_id']
    ts = datetime.fromisoformat(tx['timestamp'])

    user_events[user_id].append(ts)

    while user_events[user_id] and (ts - user_events[user_id][0]).total_seconds() > 60:
        user_events[user_id].popleft()

    if len(user_events[user_id]) > 3:
        print(f"ALERT: {user_id} wykonał {len(user_events[user_id])} transakcje w ciągu 60 sekund")

Writing consumer_speed.py
